# PolicySpace Manual

PolicySpace is a game about economies and policy. It is being built one system at a time, and so far the first system exists: **the world**. A *setting* describes a world (its goods, its map, and its natural resources), the engine loads it and runs turns, and a developer tool lets you look at the result. There is no player-facing game yet, so this manual is written for **setting authors** and developers.

*This manual grows as each part of the project is completed.*

## Contents

- [Project overview](#Project-overview)
- [Writing a setting](#Writing-a-setting)
  - [The folder layout](#The-folder-layout)
  - [Naming rules](#Naming-rules)
  - [Goods](#Goods)
  - [The map](#The-map)
  - [Natural resources](#Natural-resources)
  - [Running a setting](#Running-a-setting)
  - [Checking a setting](#Checking-a-setting)
  - [The developer UI](#The-developer-UI)
- [Technical documentation](#Technical-documentation)

## Project overview

The project is a Rust workspace with two parts: an **engine** library, which holds the rules and runs the world, and a **developer UI** program, which loads settings and shows them. The engine never depends on the developer UI. Details are in [docs/workspace.md](docs/workspace.md), and the way names become internal IDs is explained in [docs/ids.md](docs/ids.md).

Two commands matter day to day, both run from the project's root folder:

- `cargo run -p policyspace-dev-ui -- <folder>` opens the developer UI on a setting (see [The developer UI](#The-developer-UI)).
- `python tools/check.py` runs formatting checks, lint checks and all the tests.

## Writing a setting

A **setting** is a folder of plain-text [TOML](https://toml.io) files that describes a world: what goods exist, what the map looks like, and where natural resources lie. The engine has no built-in content, so everything comes from the setting. A complete example lives in `settings/sample/`; copy it to start your own.

For the technical side of the format, see [docs/setting-loader.md](docs/setting-loader.md).

### The folder layout

```
my_setting/
  setting.toml      the manifest
  rules/            things that never change during play (goods, resource types)
  start/            what turn 0 looks like (the map, the deposits)
```

**The manifest, `setting.toml`,** has these keys:

| Key | Required | Meaning |
|---|---|---|
| `name` | yes | The setting's name |
| `description` | no | A sentence about it |
| `format_version` | yes | Always `1` for now |

A minimal manifest:

```toml
name = "Sample Setting"
description = "The project's sample setting."
format_version = 1
```

Unknown keys are errors everywhere in a setting, not just in the manifest. A misspelled key is reported instead of silently ignored.

**Files and folders.**

- Organize files however you like, nested to any depth. Files are read in sorted path order.
- What a file defines comes from its section headers (such as `[[good]]`), not from its file name.
- `README.md` files are the place for notes. Hidden files, `Thumbs.db` and `desktop.ini` are ignored. Any other file that does not end in `.toml` is an error.
- Each section type belongs in one of the two folders: `[[good]]` and `[[resource]]` in `rules/`, `[[hex]]` and `[[deposit]]` in `start/`. Putting one in the wrong folder is an error.

### Naming rules

Names of goods and resource types use lowercase letters, digits and underscores, and must start with a letter: `copper_ore` and `grain2` are fine; `Copper`, `2grain` and `copper-ore` are not.

### Goods

A **good** is anything an actor can own. The engine has no built-in goods and no built-in categories such as labor or capital: the setting defines them all.

Define goods with `[[good]]` sections in `rules/`, in any file at any depth, as many per file as you like. Goods from all files are merged. A setting with no goods is valid. A `[[good]]` in `start/` is an error.

| Key | Required | Meaning |
|---|---|---|
| `name` | yes | Lowercase snake_case, unique across the whole setting |
| `perishes_after` | no | A whole number, 1 or more. Leave it out and the good never perishes. |
| `movable` | no | `true` (the default) or `false`. **It has no effect yet**; movement comes in a later stage. |

`perishes_after` counts the turn a unit appears in. A unit appearing in turn 5 with `perishes_after = 1` is gone when turn 5 ends; with `perishes_after = 3` it is gone at the end of turn 7.

```toml
# Labor: gone at the end of the turn, and tied to where it is.
[[good]]
name = "unskilled_labor"
perishes_after = 1
movable = false

# Only a name is needed: durable and movable.
[[good]]
name = "copper"
```

The sample setting defines five goods:

| Name | `perishes_after` | `movable` | Role |
|---|---|---|---|
| `unskilled_labor` | 1 | false | Labor (used in later stages) |
| `unimproved_copper` | 1 | false | What copper deposits yield |
| `unimproved_grain` | 1 | false | What arable land yields |
| `copper` | never | true | A durable, movable good |
| `grain` | 3 | true | A perishable, movable good |

More detail: [docs/goods.md](docs/goods.md).

### The map

The **map** is the set of hex tiles in the setting's starting state. For now a hex has only a position; terrain and other attributes come later. The map goes in `start/`, which describes what turn 0 looks like. A `[[hex]]` in `rules/` is an error.

Each `[[hex]]` has exactly two required whole-number keys, `q` and `r`. Any other key (such as `terrain`) is an error. Hexes have no names: the coordinates are the identity, and two hexes at the same coordinates is an error that names both places. Hexes can sit in any file at any depth and are merged.

```toml
[[hex]]
q = 0
r = 0

[[hex]]
q = 1
r = 0
```

**Coordinates and orientation.** The engine fixes the orientation; a setting cannot change it. Hexes are *pointy-top*. Each value of `r` is one straight east-west row; `q` increases eastward and `r` increases southward. So `(1, 0)` is the east neighbor of `(0, 0)`, and the row `r = 1` lies south of the row `r = 0`. Values may be negative, up to about plus or minus 2.1 billion.

**Shape.** There are no shape rules. Holes, islands and negative coordinates are all fine, and a setting with no hexes is valid. Neighbors, distance and wrapping around the edges do not exist yet; wrapping is planned for later as an optional per-setting value.

The sample map is a hexagon of radius 2 around `(0, 0)`: 19 hexes. The diagram below shows it, with each hex labelled `q, r`.

More detail: [docs/map.md](docs/map.md).

In [ ]:
import math
from IPython.display import SVG, display

def hex_svg(deposits=None, size=36):
    deposits = deposits or {}
    cells = [(q, r) for r in range(-2, 3) for q in range(-2, 3) if abs(q + r) <= 2]
    pts = [(size * math.sqrt(3) * (q + r / 2), size * 1.5 * r) for q, r in cells]
    minx = min(p[0] for p in pts) - size; maxx = max(p[0] for p in pts) + size
    miny = min(p[1] for p in pts) - size; maxy = max(p[1] for p in pts) + size
    w, h = maxx - minx, maxy - miny + 24
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w:.0f}" height="{h:.0f}" viewBox="{minx:.0f} {miny - 24:.0f} {w:.0f} {h:.0f}" font-family="sans-serif">']
    out.append(f'<text x="{minx + 4:.0f}" y="{miny - 8:.0f}" font-size="12">q grows east, r grows south</text>')
    for (q, r), (x, y) in zip(cells, pts):
        corners = " ".join(
            f"{x + size * math.cos(math.radians(60 * i - 30)):.1f},{y + size * math.sin(math.radians(60 * i - 30)):.1f}"
            for i in range(6))
        fill = "#fdf3d0" if (q, r) in deposits else "#e8f0fa"
        out.append(f'<polygon points="{corners}" fill="{fill}" stroke="#456" stroke-width="1.2"/>')
        out.append(f'<text x="{x:.1f}" y="{y - 2:.1f}" font-size="11" text-anchor="middle">{q}, {r}</text>')
        for i, line in enumerate(deposits.get((q, r), [])):
            out.append(f'<text x="{x:.1f}" y="{y + 9 + 10 * i:.1f}" font-size="8.5" text-anchor="middle" fill="#a40">{line}</text>')
    out.append("</svg>")
    return SVG("\n".join(out))

display(hex_svg())

### Natural resources

Natural resources come in two parts: **resource types** say what kinds of resource exist and what one deposit produces, and **deposits** put them on the map. Each deposit produces its type's output every turn (see [Running a setting](#Running-a-setting)). There is no ownership yet (that is a later stage), and no stock or depletion.

**Resource types** are `[[resource]]` sections in `rules/` (in `start/` it is an error), in any file at any depth, merged across files. Three keys are required:

| Key | Meaning |
|---|---|
| `name` | Lowercase snake_case, unique among resource types. It may match a good's name. |
| `produces` | The name of a good the setting defines |
| `output` | How much of that good one deposit produces per turn: a whole number or decimal, greater than 0 |

Each type produces exactly one good, but several types may produce the same one. By convention the produced good is unmovable and perishes after one turn (like the sample's `unimproved_*` goods), but the engine does not require that.

**Deposits** are `[[deposit]]` sections in `start/` (in `rules/` it is an error). Keys:

| Key | Meaning |
|---|---|
| `resource` | The name of a resource type |
| `q`, `r` | A hex that exists on the map |
| `count` | Optional, 1 to 1,000,000, default 1 |

`count = 3` makes three separate deposits. Each will be ownable by a single owner later, so the total on a hex is `count` times `output`. A hex can hold several resource types, but each type may appear only once per hex: raise `count` instead.

```toml
# rules/resources.toml
[[resource]]
name = "copper_ore"
produces = "unimproved_copper"
output = 2.5

# start/deposits.toml
[[deposit]]
resource = "copper_ore"
q = 1
r = -1
count = 2

[[deposit]]
resource = "copper_ore"
q = -2
r = 1
```

A resource's `produces`, a deposit's `resource`, and a deposit's hex are checked after all files are read, so the target can be in any file. These reference errors are not shown if part of the setting could not be read (see [Checking a setting](#Checking-a-setting)).

**The sample setting** has two resource types:

| Name | Produces | Output per deposit |
|---|---|---|
| `copper_ore` | `unimproved_copper` | 2.5 |
| `arable_land` | `unimproved_grain` | 4 |

and six deposit entries that place 10 deposits:

| Hex `(q, r)` | Resource | Count |
|---|---|---|
| `(0, 0)` | `arable_land` | 3 |
| `(0, 0)` | `copper_ore` | 1 |
| `(-1, 0)` | `arable_land` | 2 |
| `(1, 0)` | `arable_land` | 1 |
| `(1, -1)` | `copper_ore` | 2 |
| `(-2, 1)` | `copper_ore` | 1 |

Hex `(0, 0)` holds two resource types at once. In the diagram, hexes with deposits are shaded.

In [ ]:
display(hex_svg({
    (0, 0): ["arable_land x3", "copper_ore x1"],
    (-1, 0): ["arable_land x2"],
    (1, 0): ["arable_land x1"],
    (1, -1): ["copper_ore x2"],
    (-2, 1): ["copper_ore x1"],
}))

Planned for later: stock, depletion and regeneration as optional data on resource types, so settings written now stay valid.

More detail: [docs/resources.md](docs/resources.md), with [docs/map.md](docs/map.md) for coordinates and [docs/goods.md](docs/goods.md) for goods.

### Running a setting

A loaded setting becomes a running **world**. The rules stay fixed; the world holds the clock (starting at turn 0) and the goods that exist. Each turn has three parts:

1. **Emit.** At the start of the turn every deposit produces its resource type's `output` of its good.
2. **Actors act.** Nothing acts yet, so this part is empty.
3. **Perish.** At the end of the turn, goods that have reached their limit are removed.

The world is observed after the emit and before the perish. Deposits are unowned, so everything lands in an "unowned goods" stockpile on the hex where the deposit sits.

`perishes_after` takes effect here. With 1, output is usable during the turn it appears and gone when it ends. With 3, it is gone at the end of the third turn it exists. Goods without `perishes_after` last forever and accumulate. (`movable` still has no effect.)

A **stockpile** holds the total of each good, and for each total, how much perishes at the end of which turn. For example, a good with `perishes_after = 3` produced at 5 per turn:

| Turn | Held during the turn (after the emit) | Removed when the turn ends |
|---|---|---|
| 0 | 5 | 0 |
| 1 | 10 | 0 |
| 2 | 15 | 5 (the batch from turn 0) |
| 3 | 15 | 5 (the batch from turn 1) |
| 4 | 15 | 5 (the batch from turn 2) |

**Turn reports.** Each turn produces a report with totals per good, over the whole map, of what perished (when the previous turn ended) and what was emitted. The sample setting over turns 0 to 2 (see the next section for the command) looks like this:

In [ ]:
print("Turn 0: perished nothing; emitted 10 unimproved_copper, 24 unimproved_grain")
print("Turn 1: perished 10 unimproved_copper, 24 unimproved_grain; emitted 10 unimproved_copper, 24 unimproved_grain")
print("Turn 2: perished 10 unimproved_copper, 24 unimproved_grain; emitted 10 unimproved_copper, 24 unimproved_grain")

The four copper deposits give 4 x 2.5 = 10 and the six grain deposits give 6 x 4 = 24 each turn. Both goods last one turn, so each turn's output is gone before the next turn's arrives.

More detail: [docs/world.md](docs/world.md).

### Checking a setting

The checker loads a setting and reports every problem in one go, with file, line and column. It never produces a half-loaded setting.

```
dev-ui check <folder> [--turns N]
```

From the project root, that is `cargo run -p policyspace-dev-ui -- check settings/sample`. On success it prints one line:

```
Loaded setting 'Sample Setting' (format version 1): 5 goods, 19 hexes, 2 resources, 10 deposits
```

(Counts use singular wording where it applies: `1 hex`, `1 resource`, `1 deposit`.) With `--turns N` (N from 0 to 10,000) it then prints one report line per turn, from turn 0 through turn N; `--turns 0` prints only turn 0.

Problems look like this:

```
rules/a.toml:3:18: key 'perishes_after' in good must be at least 1
rules/a.toml:6:6: duplicate good name 'copper' (first defined at rules/a.toml:2:6)
rules/a.toml:10:10: unknown good 'gold'
start/m.toml:4:1: unknown key 'terrain' in hex
start/m.toml:6:1: duplicate hex at (0, 0) (first defined at start/m.toml:1:1)
5 error(s) found in my_setting
```

If part of the setting could not be read (a missing folder, or a broken or unreadable file), reference errors such as `unknown good` are not shown; fix the structural problem first.

| Exit code | Meaning |
|---|---|
| 0 | The setting loaded |
| 1 | The setting has errors |
| 2 | Wrong usage (a three-line usage message is printed) |

The checker never opens a window, so it is the form to use in scripts. `dev-ui --version` prints the engine's name and version. More detail: [docs/setting-loader.md](docs/setting-loader.md), and [docs/world.md](docs/world.md) for `--turns`.

### The developer UI

The developer UI is a window that shows a setting's map and lets you step through turns and inspect the world.

- `dev-ui <folder>` opens the window with a setting loaded.
- `dev-ui` alone opens it with an **Open setting...** button (a system folder picker; cancelling changes nothing).

For example: `cargo run -p policyspace-dev-ui -- settings/sample`.

#### Reading the map

Each hex is a pointy-top hexagon, rows run east-west, `q` increases eastward and `r` increases southward, exactly as in [The map](#The-map). The sample setting shows as a hexagon of 19 hexes. Hexes with at least one deposit show a small mark when drawn as hexagons, so you can see where to click.

**Controls**

- Drag to pan; scroll to zoom toward the cursor; **Fit** shows the whole map again.
- When zoomed in far enough, each hex shows its coordinates as `q, r` (the center of the sample is `0, 0`).
- Hovering highlights a hex and shows its coordinates in the status bar.
- The status bar also shows the setting name, the hex count and the current turn.

Unusual maps are handled: an empty map shows "This setting has no hexes."; holes, islands and negative coordinates draw correctly; for maps spanning huge distances, hexes too small to see are drawn as dots, and zooming in turns them back into hexagons.

#### Inspecting a hex

Click a hex to select it (a thick accent-colored outline, distinct from the hover highlight). When hexes are drawn as dots, a click near a dot selects it. Click empty space or press Esc to clear. Dragging to pan never changes the selection.

The **hex panel** on the right shows, for the selected hex:

- its coordinates;
- its deposits grouped by type, for example `arable_land x 3: 4 unimproved_grain each, 12 total`;
- its stockpile: each good's total, and the parts of it with a countdown to when they perish: `perishes this turn`, `perishes in N turns`, or `never perishes`.

Reading the countdown: suppose hex `(0, 0)` holds 15 of a good that lasts 3 turns, at turn 2, with 5 produced each turn. The panel lists 5 that perish this turn, 5 that perish in 1 turn and 5 that perish in 2 turns: the oldest batch is about to go, the newest has two more turns to live.

Other messages: "Click a hex to inspect it.", "No deposits.", "Empty." and "This setting has no hexes."

#### Stepping turns

The **Step** and **Step x10** buttons in the top bar advance the world one or ten turns (shortcuts: Space and Shift+Space). They work whenever the setting loaded without errors, even one with no hexes. The status bar's turn follows. The world starts at turn 0 on every load.

#### The world panel

The **world panel** on the left has three sections, which collapse; the panels can be resized.

- **Turn log:** every turn report since the load, turn 0 included, newest first, in the same wording as `check --turns` (see [Running a setting](#Running-a-setting)).
- **Goods:** each good's name, how long it lasts, whether it is movable or fixed, and the amount held in the whole world.
- **Resource types:** each type's name, the good it produces, its output per deposit and its deposit count.

#### The author's loop

1. Edit a setting file (for example, change some deposits or goods).
2. Click **Reload**. The world returns to turn 0, with a log of one line. Your pan and zoom stay, and so does your selection if the reloaded map still has a hex at those coordinates.
3. Step a few turns and check the stockpiles.

If the setting has errors, the window shows the full error list (the same lines the checker prints) and no map or side panels, and the Step buttons are disabled. Fix the files and Reload again. A failed Reload doesn't lose the selection. A broken folder given on the command line opens the window showing its errors. Opening a different folder clears the selection and the log.

#### Not available yet

Auto-run, stepping by another number, coloring hexes by data, editing the world, and selecting several hexes.

More detail: [docs/dev-ui.md](docs/dev-ui.md), with [docs/world.md](docs/world.md) for turns and stockpiles.

## Technical documentation

For developers who want to understand or modify the project, see the [technical documentation](docs/README.md).